# HELIOS — Fine-tune LLM sur les 120 incidents (Colab GPU)

Le fine-tune ne peut pas tourner sur .161 (pas de GPU). Ce notebook le lance sur le **GPU T4 gratuit de Colab**.

**Runtime → Modifier le type d'exécution → T4 GPU**, puis Exécuter tout.

À la fin : télécharge `aiops-sre-120.gguf` et fais `ollama create aiops-sre-ft-v3` sur .161.

In [ ]:
!pip -q install "transformers>=4.44" "peft>=0.12" "trl>=0.9" datasets accelerate bitsandbytes
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'AUCUN — active le T4 !')

In [ ]:
# corpus des 120 (généré depuis le catalogue HELIOS + données publiques K8s)
!wget -q -O corpus_120_sft.jsonl https://raw.githubusercontent.com/SALHIHoussam/AIOPS/helios-camunda-node-remediation/scripts/train-120/corpus_120_sft.jsonl
import json
n = sum(1 for _ in open('corpus_120_sft.jsonl'))
print('exemples SFT:', n)

In [ ]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig
from trl import SFTTrainer

BASE = 'Qwen/Qwen2.5-0.5B-Instruct'
tok = AutoTokenizer.from_pretrained(BASE)
ds = load_dataset('json', data_files='corpus_120_sft.jsonl', split='train')
ds = ds.map(lambda ex: {'text': tok.apply_chat_template(ex['messages'], tokenize=False)}, remove_columns=ds.column_names)

model = AutoModelForCausalLM.from_pretrained(BASE, device_map='auto', torch_dtype='bfloat16')
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type='CAUSAL_LM',
                  target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
args = TrainingArguments(output_dir='out', num_train_epochs=3, per_device_train_batch_size=8,
                         gradient_accumulation_steps=2, learning_rate=2e-4, warmup_ratio=0.03,
                         logging_steps=20, bf16=True, lr_scheduler_type='cosine', report_to='none')
trainer = SFTTrainer(model=model, args=args, train_dataset=ds, peft_config=lora,
                     dataset_text_field='text', max_seq_length=1024)
trainer.train()
trainer.save_model('aiops-sre-120-lora')

In [ ]:
# fusion LoRA -> modèle complet -> GGUF pour Ollama
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype='bfloat16')
merged = PeftModel.from_pretrained(base, 'aiops-sre-120-lora').merge_and_unload()
merged.save_pretrained('aiops-sre-120-merged'); tok.save_pretrained('aiops-sre-120-merged')
!git clone -q https://github.com/ggerganov/llama.cpp && pip -q install -r llama.cpp/requirements.txt
!python llama.cpp/convert_hf_to_gguf.py aiops-sre-120-merged --outfile aiops-sre-120.gguf --outtype q8_0
from google.colab import files; files.download('aiops-sre-120.gguf')

## Déploiement sur .161 (après téléchargement du .gguf)
```bash
scp aiops-sre-120.gguf kube@192.168.148.161:/tmp/
ssh kube@192.168.148.161
printf 'FROM /tmp/aiops-sre-120.gguf\nPARAMETER temperature 0.15\n' > /tmp/MF
ollama create aiops-sre-ft-v3 -f /tmp/MF   # le LLM VRAIMENT fine-tuné sur les 120
```
Puis pointer le service IA vers `aiops-sre-ft-v3`.